# Get current price data to test the model on real stocks

In [1]:
import numpy as np
import yfinance as yf
from sklearn.covariance import LedoitWolf
from scipy.stats import gmean


Selecting 8 stocks in the sp500 and calculating their covariance using the LedoitWolf method

In [2]:
start = '2023-09-24'
end = '2026-03-04'
tickers = ['AAPL', 'TSLA', 'LLY', 'BRK-B', 'V', 'BAC', 'WFC', 'IBM']

# download the data
prices = yf.download(tickers, start, end, auto_adjust=False)['Adj Close']

# 1. Prepare your data (dropping NAs is crucial for the solver)
log_returns = np.log(prices / prices.shift(1)).dropna()

# 2. Initialize the Ledoit-Wolf estimator
lw = LedoitWolf()

# 3. Fit the model to your returns 
# Note: sklearn expects (observations, features), which matches your log_returns layout
lw.fit(log_returns)

# 4. Extract the shrunk covariance matrix
# This is the "cleaner" version of your cov_matrix
shrunk_cov = lw.covariance_

# 5. Annualize (same as your original code)
shrunk_cov_annual = shrunk_cov * 252

[*********************100%***********************]  8 of 8 completed


In [10]:
# get current prices
S0 = prices.iloc[-1, :]
S0 = np.array(S0)

# get geometric average of the stock prices
k = gmean(S0)

print(f'Strike: {k}')

Strike: 252.05968491414146


Get the dividend yields for each stock

In [4]:
yields = []
for t in tickers:
    info = yf.Ticker(t).info
    # Use trailingAnnualDividendYield or dividendYield
    y = info.get('dividendYield', 0) 
    yields.append(y if y is not None else 0)

yields[-1] = 2.68 # for some reason yfinance didn't get the right divident yield for IBM...


Create parameter dictionary

In [11]:
r = 0.0375             # current BOE rate
div = np.mean(yields)
T = 0.25
d = len(tickers)




p = {}
p['strike'] = k
p['rate'] = r 
p['dividend'] = div
p['expiration'] = T
p['dim'] = d
p['S0'] = S0
p['volatility'] = None
p['correlation'] = None
p['covariance'] = shrunk_cov_annual
p['numTimeStep'] = 50
p['callput'] = 'put'

Run the G-LSM model to estimate the price of such an option

In [6]:
# Import the model
import glsm_geobasketcall

# Call the main function
glsm_geobasketcall.main(p)

Number of basis functions: 361
---------------------------------------------
run trial no.1, price = 60.7380
---------------------------------------------
run trial no.2, price = 61.0251
---------------------------------------------
run trial no.3, price = 60.5361
---------------------------------------------
run trial no.4, price = 60.7203
---------------------------------------------
run trial no.5, price = 60.5288
---------------------------------------------
run trial no.6, price = 60.9150
---------------------------------------------
run trial no.7, price = 61.0621
---------------------------------------------
run trial no.8, price = 61.0588
---------------------------------------------
run trial no.9, price = 60.7366
---------------------------------------------
run trial no.10, price = 60.8459
---------------------------------------------

Mean price: 60.8167
Std dev:    0.1888


Now price the call, we check the number with the put call parity

In [ ]:
p = {}
p['strike'] = k
p['rate'] = 0.0375 # current BOE rate
p['dividend'] = np.mean(yields)
p['expiration'] = 0.25
p['dim'] = len(tickers)
p['S0'] = S0
p['volatility'] = None
p['correlation'] = None
p['covariance'] = shrunk_cov_annual
p['numTimeStep'] = 50
p['callput'] = 'call'

In [8]:
# Import the model
import glsm_geobasketcall

# Call the main function
glsm_geobasketcall.main(p)

Number of basis functions: 361
---------------------------------------------
run trial no.1, price = 1.2847
---------------------------------------------
run trial no.2, price = 1.3172
---------------------------------------------
run trial no.3, price = 1.3065
---------------------------------------------
run trial no.4, price = 1.3102
---------------------------------------------
run trial no.5, price = 1.3240
---------------------------------------------
run trial no.6, price = 1.3021
---------------------------------------------
run trial no.7, price = 1.3017
---------------------------------------------
run trial no.8, price = 1.3001
---------------------------------------------
run trial no.9, price = 1.3038
---------------------------------------------
run trial no.10, price = 1.3002
---------------------------------------------

Mean price: 1.3050
Std dev:    0.0101


### Risk-Neutral Pricing of Geometric Mean Options

We assume that, under the risk-neutral probability measure:
$$dS_i(t) = S_i(t)(rdt + \sigma_i dW_i(t)), \quad \text{for } i=1, \dots, n$$

Where:
* **$r$** is the interest rate.
* **$\sigma_i$** is the volatility.
* **$\delta$** is the dividend rate.
* **$\{W_i(t), t \geq 0\}$** is a standard Brownian motion such that $d\langle W_i, W_j \rangle(t) = \rho_{i,j}dt$ for $i \neq j$.

Then, the geometric mean is given by:
$$\left(\prod_{i=1}^{n} S_i(T)\right)^{\frac{1}{n}} = \left(\prod_{i=1}^{n} S_i(0)\right)^{\frac{1}{n}} \exp\left( \left(r - \delta - \frac{1}{2n} \sum_{i=1}^{n} \sigma_i^2\right)T + \frac{1}{n} \sum_{i=1}^{n} \sigma_i W_i(T) \right)$$

Let the aggregate volatility $\sigma$ and the standard normal variable $\xi$ be defined as:
$$\sigma = \sqrt{\frac{1}{n^2} \left( \sum_{i=1}^{n} \sigma_i^2 + 2\sum_{i < j} \rho_{i,j} \sigma_i \sigma_j \right)}, \quad \xi = \frac{\frac{1}{n} \sum_{i=1}^{n} \sigma_i W_i(T)}{\sigma \sqrt{T}}$$

Since we have a covariance matrix ($Sigma$) instead of volatility ($\sigma_i$) and correlations ($\rho_{i,j}$), we can use the elemnts of the covariance matrix are defined as $\Sigma_{i,j} = \sum_{i,j}\sigma_i\sigma_j\phi_{i,j}$ to rewrite the equation for $\sigma$ as

$$\sigma = \sqrt{\frac{1}{n^2}\sum_{i=1}^{n}\sum_{j=1}^{n}\Sigma_{i,j}}.$$

Then, $\xi \sim N(0,1)$. Let:
$$F = \left(\prod_{i=1}^{n} S_i(0)\right)^{\frac{1}{n}} \exp\left( \left(r - \frac{1}{2n} \sum_{i=1}^{n} \sigma_i^2 + \frac{1}{2}\sigma^2\right)T \right)$$

The geometric mean at time $T$ can be expressed as:
$$\left(\prod_{i=1}^{n} S_i(T)\right)^{\frac{1}{n}} = F e^{-\frac{1}{2}\sigma^2 T + \sigma \sqrt{T} \xi}$$

Following the derivation of the **Black-Scholes formula**, the expected payoff is:
$$E\left[\max\left(\left(\prod_{i=1}^{n} S_i(T)\right)^{\frac{1}{n}} - K, 0\right)\right] = F\Phi(d_1) - K\Phi(d_2)$$

Where $\Phi$ is the standard normal CDF, and:
$$d_1 = \frac{\ln(F/K) + \frac{1}{2}\sigma^2 T}{\sigma \sqrt{T}}, \quad d_2 = d_1 - \sigma \sqrt{T}$$

The option value is then:
$$C = e^{-rT} [F\Phi(d_1) - K\Phi(d_2)].$$

Using the put-call parity, we can derive the expression for the price of the geometric put option.

Put-call partiy:
$$C - P = e^{-rT}(F-K)$$

Rearranging for P, and substituting in C from above we find

\begin{align}
P &= e^{-rT} [F\Phi(d_1) - K\Phi(d_2)] - e^{-rT}(F-K)\\
P &= e^{-rT}[K(1-\Phi(d_2)) - F(1-\Phi(d_1))].
\end{align}

We now use the fact that $1-\Phi(a) = \Phi(-a)$ to find the value of the put option is

$$P=e^{-rT}[K\Phi(-d_2) - F\Phi(-d_1)].

In [23]:
# used to find the cumulative distribution values
from scipy.stats import norm

# calculating the price of the call and the put with put call parity

# breaking up the calculations
sigmas_sqr = np.diag(shrunk_cov_annual)
sum_sig_sqr = np.sum(sigmas_sqr)
sigma = np.sqrt(1/d**2 * np.sum(shrunk_cov_annual))


# term in the exponential for F
exp_term = (r - div - 1/(2*d)*sum_sig_sqr + 0.5*sigma**2) * T

# calculate F
F = k * np.exp(exp_term)

# calculate d1 and d2
d1 = (np.log(F/k) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma*np.sqrt(T)

# calcualte the values from the cumulative distribution function
phi_d1 = norm.cdf(d1); phi_md1 = norm.cdf(-d1)
phi_d2 = norm.cdf(d2); phi_md2 = norm.cdf(-d2)

# prices
C = np.exp(-r*T) * (F*phi_d1 - k*phi_d2)
P = np.exp(-r*T) * (k*phi_md2 - F*phi_md1)

print(f'Call price: {C}')
print(f'Put price: {P}')


Call price: 0.007533241930630897
Put price: 60.852171232609535


# Error in G-LSM method

In [25]:
glsm_call_price = 1.3050
glsm_put_price = 60.8167

error_call = abs(C - glsm_call_price) / abs(C)
error_put = abs(P - glsm_put_price) / abs(P)

print(f'Call error: {error_call*100:.2}%')
print(f'Put error: {error_put*100:.2}%')


Call error: 1.7e+04%
Put error: 0.058%
